# Age & Gender Model Evaluation
This notebook processes your training metrics CSV and plots the loss. It also loads your trained model for inference on a single image.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load and clean the CSV
# Update this path to point to your actual generated metrics.csv
csv_path = "../mobilenet_v3_large_aug_metrics.csv"
df = pd.read_csv(csv_path)

# Forward fill and backward fill to merge train/val rows per epoch if needed
df_clean = df.groupby('epoch').apply(lambda x: x.bfill().ffill().iloc[0]).reset_index(drop=True)

# 2. Plot Training vs Validation Loss
plt.figure(figsize=(10, 6))
plt.plot(df_clean['epoch'], df_clean['train_loss'], label='Train Loss', marker='o')
plt.plot(df_clean['epoch'], df_clean['val_loss'], label='Val Loss', marker='o')
plt.title('Training and Validation Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.legend()
plt.grid(True)
plt.show()

## Single Image Inference
Load a saved checkpoint and perform inference.

In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import sys
import os

# Ensure the src module can be imported from the root folder
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.runners.trainer import load_model

# 1. Load the model from model_store
# Ensure you provide the exact path inside model_store/
checkpoint_name = "your_model_best.pth" 
model = load_model(checkpoint_name)
model.eval()
model.to('cuda' if torch.cuda.is_available() else 'cpu')
print("Model loaded successfully!")

# 2. Define Image Transforms
# Using standard ImageNet normalization
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_image(image_path, model):
    img = Image.open(image_path).convert("RGB")
    
    # Show the image
    plt.imshow(img)
    plt.axis('off')
    plt.show()
    
    # Preprocess and prepare for model
    img_tensor = val_transforms(img).unsqueeze(0).to(model.device)
    
    # Inference
    with torch.no_grad():
        age_pred, gender_logits = model(img_tensor)
        
        # Process outputs
        age = age_pred.item()
        # Gender: output is logits for [Male, Female]
        gender_prob = torch.softmax(gender_logits, dim=1)
        gender_pred = torch.argmax(gender_prob, dim=1).item()
        
        gender_label = "Female" if gender_pred == 1 else "Male"
        
        print(f"Prediction -> Age: {age:.1f} years | Gender: {gender_label} (Confidence: {gender_prob[0][gender_pred].item():.2f})")

# Run test on an image
test_image_path = "../test-images/face5.jpg" # Update with an actual image path
if os.path.exists(test_image_path):
    predict_image(test_image_path, model)
else:
    print(f"Please update test_image_path to point to a valid image. Currently set to: {test_image_path}")